# 📚 Support Vector Machines (SVM) — Complete Lecture
### No heavy math — just intuition, visuals, and code

---

## 🗺️ What We Will Cover Today

| Section | Topic |
|---|---|
| 1 | What problem does SVM solve? |
| 2 | The idea of a margin |
| 3 | Support Vectors — what are they? |
| 4 | Hard Margin vs Soft Margin |
| 5 | The C parameter — explained intuitively |
| 6 | Bias and Variance in SVMs |
| 7 | When data is NOT linearly separable — The Kernel Trick |
| 8 | The RBF Kernel and the Gamma parameter |
| 9 | Polynomial Kernel |
| 10 | Comparing all kernels |
| 11 | Real dataset walkthrough |
| 12 | Summary cheat sheet |

In [ ]:
# ── Run this first — loads everything we need ─────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn import svm, datasets
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
from sklearn.datasets import make_moons, make_circles, make_classification

np.random.seed(42)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

COLORS = ['#E63946', '#457B9D']
print('✅ Libraries loaded — ready to go!')

---
## Section 1 — What Problem Does SVM Solve?

Imagine you have two groups of points on a piece of paper — red dots and blue dots.  
You want to draw a **line** that separates them so that:
- All reds are on one side
- All blues are on the other side

**The problem:** There are *infinite* lines that can separate them perfectly. Which one do you pick?

👉 **SVM's answer:** Pick the line that is **as far away as possible from both groups.**  
That gap is called the **margin** — and SVM tries to **maximise it**.

Why does this matter? A wider margin means the model is more confident and less likely to be fooled by new, slightly different data points.

In [ ]:
# ── Visualising the problem: many lines separate, SVM picks the BEST one ──
np.random.seed(1)
X_demo = np.r_[np.random.randn(15,2)-[2,2], np.random.randn(15,2)+[2,2]]
y_demo = np.array([-1]*15 + [1]*15)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: show many possible separating lines
ax = axes[0]
ax.scatter(X_demo[y_demo==-1,0], X_demo[y_demo==-1,1], c=COLORS[0], s=70, edgecolors='k', label='Class -1')
ax.scatter(X_demo[y_demo== 1,0], X_demo[y_demo== 1,1], c=COLORS[1], s=70, edgecolors='k', label='Class +1')
x_line = np.linspace(-5, 5, 100)
for slope, intercept, alpha in [(-1, 0, 0.4), (0.5, 0.5, 0.4), (2, -1, 0.4)]:
    ax.plot(x_line, slope*x_line + intercept, 'gray', alpha=alpha, lw=1.5, linestyle='--')
ax.set_title('Problem: Infinite lines separate the data\nWhich one is best?', fontweight='bold')
ax.legend(); ax.set_xlim(-5,5); ax.set_ylim(-5,5)

# Right: SVM's answer — maximum margin
ax = axes[1]
clf = svm.SVC(kernel='linear', C=1e6)
clf.fit(X_demo, y_demo)
x_min, x_max = X_demo[:,0].min()-1, X_demo[:,0].max()+1
y_min2, y_max2 = X_demo[:,1].min()-1, X_demo[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min,x_max,300), np.linspace(y_min2,y_max2,300))
Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, levels=[-999,0,999], colors=['#ffc8c8','#c8d8f0'], alpha=0.4)
ax.contour(xx, yy, Z, levels=[-1,0,1], linestyles=['--','-','--'],
           colors=['#E63946','black','#457B9D'], linewidths=[1.5,2.5,1.5])
ax.scatter(X_demo[y_demo==-1,0], X_demo[y_demo==-1,1], c=COLORS[0], s=70, edgecolors='k')
ax.scatter(X_demo[y_demo== 1,0], X_demo[y_demo== 1,1], c=COLORS[1], s=70, edgecolors='k')
sv = clf.support_vectors_
ax.scatter(sv[:,0], sv[:,1], s=200, facecolors='none', edgecolors='gold', lw=2.5, label='Support Vectors ⭐')
ax.set_title('SVM Answer: The line with MAXIMUM MARGIN\n(dashed lines = margin edges)', fontweight='bold')
ax.legend()

plt.tight_layout(); plt.show()

---
## Section 2 — The Margin

Think of the margin as a **street** or **buffer zone** between the two classes.

```
   ❌  ❌              ✅  ✅
   ❌                       ✅
        |-- margin --|  
   ❌  ❌         ✅  ✅  ✅
        ↑              ↑
   margin edge    margin edge
        \              /
         decision line  (hyperplane)
```

- The **decision boundary** (solid line) sits in the **middle** of the street
- The **margin edges** (dashed lines) are the walls of the street
- The **wider the street**, the more confident the model is

**SVM's goal = make that street as wide as possible**

---
## Section 3 — What Are Support Vectors?

The dashed margin lines pass through certain data points — these are the **support vectors**.

### 🔑 Why they matter:
- They are the **only points** that determine where the boundary goes
- If you **deleted any other point**, the boundary would **not change at all**
- If you **moved a support vector**, the boundary **would move**
- They literally "support" (hold up) the margin — hence the name

### Think of it like this:
> Imagine you're building a wall between two groups of people. The wall's position is determined only by the people standing closest to it — not by everyone far away. Those closest people are the "support vectors".

In the plot above, they are shown with a **gold circle** around them. ⭐

---
## Section 4 — Hard Margin vs Soft Margin

### Hard Margin
- The model says: **"No exceptions. Every single point must be on the correct side of the margin."**
- This works **only when data is perfectly separable** (no overlap between classes)
- Real-world data almost never looks like this!
- One outlier completely breaks the model

### Soft Margin
- The model says: **"It's okay to make a few mistakes. I'll allow some points to be on the wrong side, as long as the overall boundary is still good."**
- Much more practical for real data
- We control **how tolerant** the model is with the **C parameter**

In [ ]:
# ── Hard Margin vs Soft Margin visually ───────────────────────────────────
np.random.seed(3)
# Overlapping data (NOT perfectly separable)
X_overlap = np.r_[np.random.randn(25,2)-[1.5,1.5], np.random.randn(25,2)+[1.5,1.5]]
y_overlap = np.array([-1]*25 + [1]*25)

# Perfectly separable data
X_clean = np.r_[np.random.randn(25,2)-[3,3], np.random.randn(25,2)+[3,3]]
y_clean  = np.array([-1]*25 + [1]*25)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

def show_svm(ax, X, y, C, title):
    clf = svm.SVC(kernel='linear', C=C)
    clf.fit(X, y)
    x_min,x_max = X[:,0].min()-1, X[:,0].max()+1
    y_min,y_max = X[:,1].min()-1, X[:,1].max()+1
    xx,yy = np.meshgrid(np.linspace(x_min,x_max,300), np.linspace(y_min,y_max,300))
    Z = clf.decision_function(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx,yy,Z, levels=[-999,0,999], colors=['#ffc8c8','#c8d8f0'], alpha=0.4)
    ax.contour(xx,yy,Z, levels=[-1,0,1], linestyles=['--','-','--'],
               colors=['#E63946','black','#457B9D'], linewidths=[1.5,2.5,1.5])
    ax.scatter(X[y==-1,0],X[y==-1,1],c=COLORS[0],s=60,edgecolors='k')
    ax.scatter(X[y== 1,0],X[y== 1,1],c=COLORS[1],s=60,edgecolors='k')
    sv = clf.support_vectors_
    ax.scatter(sv[:,0],sv[:,1],s=200,facecolors='none',edgecolors='gold',lw=2.5,label=f'SVs = {len(sv)}')
    ax.set_title(title, fontweight='bold'); ax.legend()

show_svm(axes[0], X_clean,   y_clean,   1e6, 'HARD Margin (C = ∞)\nOnly works when data is perfectly separable')
show_svm(axes[1], X_overlap, y_overlap, 1,   'SOFT Margin (C = 1)\nAllows some mistakes — more realistic!')

plt.tight_layout(); plt.show()

---
## Section 5 — The C Parameter: Your Tolerance Dial 🎛️

**C is the most important hyperparameter in SVM.**  
It controls how much the model **cares about misclassifying training points**.

### The Analogy: A Strict vs Lenient Examiner

| | Strict Examiner (Large C) | Lenient Examiner (Small C) |
|---|---|---|
| **Attitude** | "Every student must pass!" | "It's okay if a few fail" |
| **Result** | Tailors rules tightly to this class | Makes general rules that may miss some |
| **Problem** | Won't work for next year's students (overfitting) | Too general, misses many students (underfitting) |

### In SVM terms:

```
SMALL C                              LARGE C
─────────────────────────────────────────────────────────
"I don't mind mistakes"              "No mistakes allowed"
↓                                    ↓
Wide margin                          Narrow margin
Fewer support vectors                More support vectors
Simpler boundary                     Complex, wiggly boundary
HIGH BIAS (underfit)                 HIGH VARIANCE (overfit)
```

> **🧠 Rule of thumb:** Start with C=1. If you're underfitting, increase C. If you're overfitting, decrease C.

In [ ]:
# ── See exactly how C changes the boundary ────────────────────────────────
np.random.seed(7)
X_c, y_c = make_moons(n_samples=120, noise=0.2, random_state=7)

C_values = [0.001, 0.1, 1, 10, 1000]
fig, axes = plt.subplots(1, 5, figsize=(24, 4))

labels = [
    'Very Small C=0.001\nUNDERFITTING\n(too simple)',
    'Small C=0.1\nSlightly underfit',
    'C=1\n✅ Usually a good start',
    'Large C=10\nSlightly overfit',
    'Very Large C=1000\nOVERFITTING\n(too complex)'
]

for ax, C, label in zip(axes, C_values, labels):
    clf = svm.SVC(kernel='rbf', C=C, gamma='scale')
    clf.fit(X_c, y_c)
    x_min,x_max = X_c[:,0].min()-0.5, X_c[:,0].max()+0.5
    y_min,y_max = X_c[:,1].min()-0.5, X_c[:,1].max()+0.5
    xx,yy = np.meshgrid(np.linspace(x_min,x_max,300), np.linspace(y_min,y_max,300))
    Z = clf.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx,yy,Z,alpha=0.3,cmap=ListedColormap(COLORS))
    ax.scatter(X_c[:,0],X_c[:,1],c=[COLORS[int(i)] for i in y_c],edgecolors='k',s=30,zorder=3)
    sv = clf.support_vectors_
    ax.scatter(sv[:,0],sv[:,1],s=150,facecolors='none',edgecolors='gold',lw=2,zorder=4)
    acc = clf.score(X_c, y_c)
    ax.set_title(f'{label}\nAcc={acc:.2f}  SVs={len(sv)}', fontsize=9, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('How C Changes the Decision Boundary (RBF kernel, γ fixed)', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout(); plt.show()

### 👀 What to notice:
- **C=0.001**: The boundary is almost a straight line — too simple for this data
- **C=1**: Smooth, sensible curved boundary
- **C=1000**: The boundary twists and wraps around individual points — memorising the training data!
- The **number of support vectors decreases** as C increases
- **High training accuracy ≠ good model** — watch C=1000!

---
## Section 6 — Bias and Variance in SVMs

These two concepts describe the two ways a model can go wrong:

### Bias = "The model is too dumb"
- It makes the same kinds of mistakes over and over
- It has not learned the data well enough
- Also called **underfitting**
- Cause in SVM: **C is too small**

### Variance = "The model is too sensitive"
- It learned the training data *too* well — including the noise
- It works great on training data but poorly on new data
- Also called **overfitting**
- Cause in SVM: **C is too large**

### The Target Analogy:
```
HIGH BIAS          HIGH VARIANCE       IDEAL
(underfitting)     (overfitting)

  . . .             x   .              . .
  . . .           .   x   .            .x.
  . . .             . x .              . .

Consistently        All over           Close to
wrong place         the place          the target
```

In [ ]:
# ── Plot training accuracy vs CV accuracy as C changes ────────────────────
C_range = np.logspace(-3, 3, 25)
train_acc_list, cv_acc_list = [], []

for C in C_range:
    clf = svm.SVC(kernel='rbf', C=C, gamma='scale')
    clf.fit(X_c, y_c)
    train_acc_list.append(clf.score(X_c, y_c))
    cv_acc_list.append(cross_val_score(clf, X_c, y_c, cv=5).mean())

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(C_range, train_acc_list, 'o-', color='#E63946', label='Training accuracy')
ax.semilogx(C_range, cv_acc_list,    's-', color='#457B9D', label='Cross-validation accuracy\n(estimate of real-world performance)')

# Shade regions
ax.axvspan(C_range[0],  C_range[5],  alpha=0.12, color='blue')
ax.axvspan(C_range[18], C_range[-1], alpha=0.12, color='red')
ax.text(0.0015, 0.72, '⬆ HIGH BIAS\n  (underfit)', color='blue', fontsize=9)
ax.text(80,     0.72, 'HIGH VARIANCE ⬆\n  (overfit)',  color='red',  fontsize=9)

best_C = C_range[np.argmax(cv_acc_list)]
ax.axvline(best_C, color='green', lw=2, linestyle=':', label=f'Best C ≈ {best_C:.2f} (from CV)')

ax.set_xlabel('C  (log scale — each step is ×10)')
ax.set_ylabel('Accuracy')
ax.set_title('Bias-Variance Trade-off: How C Affects Performance', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'\nKey observation:')
print(f'  Training accuracy keeps going UP as C increases (model memorises training data)')
print(f'  CV accuracy peaks around C={best_C:.2f} then FALLS (model stops generalising)')
print(f'  The gap between train and CV accuracy = how much the model is overfitting')

### 💡 How to use this in practice:
- **Always look at cross-validation accuracy, not just training accuracy**
- If train accuracy >> CV accuracy → overfitting → **reduce C**
- If both are low → underfitting → **increase C** or try a different kernel
- We always pick the C that gives **best CV accuracy**

---
## Section 7 — When Data is NOT Linearly Separable: The Kernel Trick

### The Problem
Sometimes no straight line can separate the classes — think of a circle of red dots surrounded by blue dots.

### The Idea: Lift the data into higher dimensions

Here is the intuition with a simple 1D example:

```
  Original 1D:    ● ● ───────── ○ ○ ○ ─────── ● ●
  (Not separable — reds are in two separate clusters)

  If we add a new dimension: x² (square of x)
  
  New 2D space:   
                  high x²  |      ● ●    ● ●
                            |    
                   low x²  |         ○ ○ ○
                            |__________________________
                                low x        high x
  
  NOW a horizontal line separates them! ✅
```

### What is a Kernel?
A **kernel** is a mathematical function that **measures similarity between points**.  
The clever trick is that SVM can use this similarity measure to work in very high-dimensional spaces **without actually computing the coordinates** in that space.

This is called the **kernel trick** — we get the benefit of higher dimensions without the cost.

In [ ]:
# ── Visual proof of the kernel trick with 1D data ─────────────────────────
x1d = np.array([-3, -2.2, -1.8, -0.4, 0, 0.4, 1.8, 2.2, 3])
y1d = np.array([ 1,   1,   1,  -1, -1,  -1,  1,  1,  1])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) Original 1D — cannot separate
ax = axes[0]
ax.scatter(x1d[y1d==1],  [0]*sum(y1d==1),  c=COLORS[1], s=150, zorder=3, label='Class +1 (Blue)')
ax.scatter(x1d[y1d==-1], [0]*sum(y1d==-1), c=COLORS[0], s=150, zorder=3, label='Class -1 (Red)')
ax.axhline(0, color='gray', lw=0.5)
ax.set_yticks([]); ax.set_xlabel('x value')
ax.set_title('1D Original Space\n❌ Cannot separate with a point', fontweight='bold')
ax.legend(fontsize=9); ax.set_ylim(-0.5, 0.5)

# 2) Lifted to 2D using x² mapping
ax = axes[1]
ax.scatter(x1d[y1d==1],  x1d[y1d==1]**2,  c=COLORS[1], s=150, zorder=3)
ax.scatter(x1d[y1d==-1], x1d[y1d==-1]**2, c=COLORS[0], s=150, zorder=3)
ax.axhline(1.5, color='black', lw=2.5, linestyle='--', label='Linear boundary (now works!)')
ax.set_xlabel('x'); ax.set_ylabel('x²  (new dimension)')
ax.set_title('2D Lifted Space: φ(x) = (x, x²)\n✅ NOW linearly separable!', fontweight='bold')
ax.legend(fontsize=9)

# 3) What the boundary looks like back in original space
ax = axes[2]
ax.scatter(x1d[y1d==1],  [0]*sum(y1d==1),  c=COLORS[1], s=150, zorder=3)
ax.scatter(x1d[y1d==-1], [0]*sum(y1d==-1), c=COLORS[0], s=150, zorder=3)
ax.axvline(-np.sqrt(1.5), color='black', lw=2.5, linestyle='--')
ax.axvline( np.sqrt(1.5), color='black', lw=2.5, linestyle='--', label='Boundary in original space')
ax.set_yticks([]); ax.set_xlabel('x value')
ax.set_title('Back in Original Space\nBoundary = two vertical lines (curved!)', fontweight='bold')
ax.legend(fontsize=9); ax.set_ylim(-0.5, 0.5)

plt.suptitle('The Kernel Trick: Lifting Data to Higher Dimensions', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout(); plt.show()

### The Four Main Kernels

| Kernel | What it does | Best used when |
|---|---|---|
| **Linear** | No transformation — stays in original space | Data is already (roughly) linearly separable; text/document data |
| **RBF (Gaussian)** | Maps to infinite dimensions; creates smooth curved boundaries | Default choice — works well in most cases |
| **Polynomial** | Captures feature interactions (e.g. x₁ × x₂) | When combinations of features matter |
| **Sigmoid** | Similar to neural network activation | Rarely used — mostly experimental |

---
## Section 8 — The RBF Kernel and the Gamma (γ) Parameter

### What does RBF stand for?
RBF = **Radial Basis Function**.  
It measures how similar two points are based on their **distance**.

### The intuition for Gamma:
Gamma controls **"how far a single training point's influence reaches"**

Imagine dropping a stone in water. The ripples spread outward.

- **Small Gamma** = The ripples spread **far** → each point influences a large area → smooth, wide boundary
- **Large Gamma** = The ripples spread **close** → each point only influences nearby points → tight, wiggly boundary

```
SMALL γ (e.g. 0.01)              LARGE γ (e.g. 100)
────────────────────             ──────────────────────
Each point has wide reach        Each point has tiny reach
Boundary = smooth curve          Boundary = wraps around each point
Result = may underfit            Result = may overfit
```

> **Remember:** γ (gamma) is the "zoom" of the kernel.  
> Small γ = zoomed out (smooth) | Large γ = zoomed in (detailed/wiggly)

In [ ]:
# ── Visualise gamma's effect on the boundary ──────────────────────────────
X_g, y_g = make_moons(n_samples=150, noise=0.2, random_state=42)

gamma_vals   = [0.01,  0.1,   1,    10,   100]
gamma_labels = ['Very Small\nγ=0.01\n(underfit)',
                'Small\nγ=0.1',
                'Medium\nγ=1\n✅ often good',
                'Large\nγ=10',
                'Very Large\nγ=100\n(overfit)']

fig, axes = plt.subplots(1, 5, figsize=(24, 4))

for ax, g, lbl in zip(axes, gamma_vals, gamma_labels):
    clf = svm.SVC(kernel='rbf', C=1, gamma=g)
    clf.fit(X_g, y_g)
    x_min,x_max = X_g[:,0].min()-0.5, X_g[:,0].max()+0.5
    y_min,y_max = X_g[:,1].min()-0.5, X_g[:,1].max()+0.5
    xx,yy = np.meshgrid(np.linspace(x_min,x_max,300), np.linspace(y_min,y_max,300))
    Z = clf.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx,yy,Z,alpha=0.35,cmap=ListedColormap(COLORS))
    ax.scatter(X_g[:,0],X_g[:,1],c=[COLORS[int(i)] for i in y_g],edgecolors='k',s=30,zorder=3)
    sv = clf.support_vectors_
    ax.scatter(sv[:,0],sv[:,1],s=150,facecolors='none',edgecolors='gold',lw=2,zorder=4)
    acc_tr = clf.score(X_g, y_g)
    acc_cv = cross_val_score(clf, X_g, y_g, cv=5).mean()
    ax.set_title(f'{lbl}\nTrain={acc_tr:.2f} CV={acc_cv:.2f}', fontsize=9, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('RBF Kernel — Effect of Gamma  (C=1 fixed throughout)', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout(); plt.show()

print('Notice:')
print('  γ=0.01 → straight-ish boundary (cannot capture the moon shape)')
print('  γ=1    → smooth curve that follows the moon shape')
print('  γ=100  → boundary wraps around individual points (overfitting!)')

### C vs Gamma — What Controls What?

| Parameter | Controls | Too small → | Too large → |
|---|---|---|---|
| **C** | How much you penalise mistakes | Wide margin, tolerates errors (underfit) | Narrow margin, no errors tolerated (overfit) |
| **γ (gamma)** | How far each point's influence reaches | Smooth, simple boundary (underfit) | Wiggly, complex boundary (overfit) |

**Both C and γ need to be tuned together using cross-validation.**

---
## Section 9 — Polynomial Kernel: Effect of Degree

In [ ]:
# ── Polynomial kernel degrees ─────────────────────────────────────────────
degrees = [1, 2, 3, 5, 8]
deg_labels = ['Degree 1\n(= linear!)', 'Degree 2\n(quadratic)', 'Degree 3\n(cubic)',
              'Degree 5\n(complex)', 'Degree 8\n(very complex)']

fig, axes = plt.subplots(1, 5, figsize=(24, 4))
for ax, d, lbl in zip(axes, degrees, deg_labels):
    clf = svm.SVC(kernel='poly', C=1, degree=d, coef0=1, gamma='scale')
    clf.fit(X_g, y_g)
    x_min,x_max = X_g[:,0].min()-0.5, X_g[:,0].max()+0.5
    y_min,y_max = X_g[:,1].min()-0.5, X_g[:,1].max()+0.5
    xx,yy = np.meshgrid(np.linspace(x_min,x_max,300), np.linspace(y_min,y_max,300))
    Z = clf.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx,yy,Z,alpha=0.35,cmap=ListedColormap(COLORS))
    ax.scatter(X_g[:,0],X_g[:,1],c=[COLORS[int(i)] for i in y_g],edgecolors='k',s=30,zorder=3)
    acc_tr = clf.score(X_g, y_g)
    acc_cv = cross_val_score(clf, X_g, y_g, cv=5).mean()
    ax.set_title(f'{lbl}\nTrain={acc_tr:.2f} CV={acc_cv:.2f}', fontsize=9, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Polynomial Kernel — Effect of Degree  (C=1 fixed)', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout(); plt.show()

### What degree means:
- **Degree 1** = linear kernel (a straight line / flat plane)
- **Degree 2** = can capture curves (parabola-like)
- **Degree 3** = can capture S-curves
- **Very high degree** = very wiggly, tends to overfit

Polynomial kernel is good when you believe **combinations of features** matter  
(e.g. height × weight, not just height or weight separately).

---
## Section 10 — Comparing All Kernels Side-by-Side

In [ ]:
# ── 4 kernels on 3 different datasets ─────────────────────────────────────
X_lin, y_lin = make_classification(n_samples=150, n_features=2, n_redundant=0,
                                    n_clusters_per_class=1, random_state=1)
X_moon2, y_moon2 = make_moons(n_samples=150, noise=0.2, random_state=0)
X_circ, y_circ   = make_circles(n_samples=150, noise=0.1, factor=0.4, random_state=0)

datasets_list = [
    (X_lin,   y_lin,   'Linear data'),
    (X_moon2, y_moon2, 'Moon-shaped data'),
    (X_circ,  y_circ,  'Circular data'),
]
kernels_list = [
    ('linear',  {},                            'Linear Kernel'),
    ('poly',    {'degree':3,'coef0':1},        'Polynomial (d=3)'),
    ('rbf',     {'gamma':'scale'},             'RBF Kernel'),
    ('sigmoid', {'coef0':0,'gamma':'scale'},   'Sigmoid Kernel'),
]

fig, axes = plt.subplots(3, 4, figsize=(20, 14))

for row, (X, y, dname) in enumerate(datasets_list):
    for col, (k, params, kname) in enumerate(kernels_list):
        ax = axes[row][col]
        clf = svm.SVC(kernel=k, C=1, **params)
        clf.fit(X, y)
        x_min,x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
        y_min,y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
        xx,yy = np.meshgrid(np.linspace(x_min,x_max,200), np.linspace(y_min,y_max,200))
        Z = clf.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx,yy,Z,alpha=0.3,cmap=ListedColormap(COLORS))
        ax.scatter(X[:,0],X[:,1],c=[COLORS[int(i)] for i in y],edgecolors='k',s=20,zorder=3)
        acc = clf.score(X, y)
        if row == 0: ax.set_title(kname, fontsize=12, fontweight='bold')
        if col == 0: ax.set_ylabel(dname, fontsize=11, fontweight='bold')
        ax.text(0.98, 0.02, f'Acc={acc:.2f}', transform=ax.transAxes,
                ha='right', fontsize=9, bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
        ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('All Kernels × All Dataset Types  (C=1 throughout)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

### Key Observations:
- **Linear kernel** works perfectly on linear data but fails on circles
- **RBF kernel** is a strong default — handles all three datasets reasonably well
- **Polynomial** can handle curved data if the degree is right
- **Sigmoid** is inconsistent — often not a good choice

→ **In practice: try RBF first, tune C and γ, then try others if needed.**

---
## Section 11 — Real Dataset: Breast Cancer Diagnosis

Let's apply everything on real medical data.  
The task: **Classify tumours as Malignant or Benign** using 30 features.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Load data
bc = datasets.load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(bc.data, bc.target, test_size=0.2, random_state=42)

# ⚠️ Always scale! SVM uses distances — bigger numbers dominate if not scaled
pipe = Pipeline([
    ('scaler', StandardScaler()),          # Step 1: scale all features to same range
    ('svm',    svm.SVC(kernel='rbf',       # Step 2: train SVM
                       C=10, gamma='scale'))
])

pipe.fit(X_tr, y_tr)
y_pred = pipe.predict(X_te)

print(f'Test Accuracy: {pipe.score(X_te, y_te)*100:.1f}%')
print()
print(classification_report(y_te, y_pred, target_names=bc.target_names))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_te, y_pred, display_labels=bc.target_names,
                                         ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Breast Cancer SVM', fontweight='bold')
plt.tight_layout(); plt.show()

### ⚠️ Why scaling matters — ALWAYS do it before SVM!

In [ ]:
# Compare: with vs without scaling
clf_no_scale = svm.SVC(kernel='rbf', C=10, gamma='scale')
clf_no_scale.fit(X_tr, y_tr)
acc_no_scale = clf_no_scale.score(X_te, y_te)

print(f'WITHOUT scaling: {acc_no_scale*100:.1f}%  ← Often poor!')
print(f'WITH scaling:    {pipe.score(X_te, y_te)*100:.1f}%  ← Much better!')
print()
print('Why? SVM computes distances between points.')
print('If one feature ranges 0-1000 and another ranges 0-1,')
print('the large-range feature will completely dominate the distance calculation.')
print('Scaling puts all features on the same footing.')

---
## Section 12 — Complete Summary

```
╔══════════════════════════════════════════════════════════════════╗
║                    SVM COMPLETE CHEAT SHEET                     ║
╠═════════════════╦════════════════════════════════════════════════╣
║ Concept         ║ Plain English                                  ║
╠═════════════════╬════════════════════════════════════════════════╣
║ Hyperplane      ║ The decision boundary (line in 2D)             ║
║ Margin          ║ The gap/street between the two classes         ║
║ Support Vectors ║ Points on the margin edge — they decide        ║
║                 ║ where the boundary goes                        ║
║ Hard Margin     ║ Zero tolerance for mistakes (needs clean data) ║
║ Soft Margin     ║ Allows some mistakes (real-world friendly)     ║
╠═════════════════╬════════════════════════════════════════════════╣
║ Small C         ║ Wide margin, tolerates errors → may underfit  ║
║ Large C         ║ Narrow margin, no errors → may overfit         ║
╠═════════════════╬════════════════════════════════════════════════╣
║ Kernel          ║ Maps data to higher dimensions                 ║
║ Linear Kernel   ║ No mapping — use for text/linearly-sep. data   ║
║ RBF Kernel      ║ Best default — smooth curved boundaries        ║
║ Poly Kernel     ║ Feature interactions — use degree 2 or 3       ║
╠═════════════════╬════════════════════════════════════════════════╣
║ Small Gamma (γ) ║ Wide influence → smooth boundary (may underfit)║
║ Large Gamma (γ) ║ Tight influence → wiggly boundary (may overfit)║
╠═════════════════╬════════════════════════════════════════════════╣
║ ALWAYS SCALE!   ║ Use StandardScaler before fitting SVM          ║
║ Tune with CV    ║ Use GridSearchCV to find best C and gamma      ║
╚═════════════════╩════════════════════════════════════════════════╝
```

### Practical Workflow for Any New Dataset:
1. **Scale** features with `StandardScaler`
2. **Start with RBF kernel**, `C=1`, `gamma='scale'`
3. **Cross-validate** to check for overfitting/underfitting
4. **GridSearch** over C and gamma to tune
5. **Try other kernels** if RBF isn't performing well